In [8]:
import networkx as nx
from tqdm import tqdm
import fasttext
import re
import os

# Charger ton modèle FastText personnalisé pour Assembly
path = "./Assembly_Model/fasttext_assembly_model.bin"
model = fasttext.load_model(path)

# Expressions régulières pour extraire les nœuds et les arêtes
EDGE_REGEX = re.compile(r'"([a-zA-Z0-9]+)"\s*->\s*"([a-zA-Z0-9]+)"')
NODE_REGEX = re.compile(r'"([a-zA-Z0-9]+)"\s*\[label\s*=\s*"(.*?)"\s*\]')

# Cache pour éviter de recalculer les embeddings des instructions répétées
embedding_cache = {}

# Expressions régulières pour nettoyer l'assembly
MNEMONIC_CLEANER = re.compile(r'[^a-zA-Z]')
REGISTER_REGEX = re.compile(r'\b(e?[abcd]x|e?[sd]i|e?[bs]p|r[0-9]+[dwb]?|xmm[0-9]+|ymm[0-9]+|zmm[0-9]+)\b', re.IGNORECASE)

# Fonction pour parser un fichier CFG
def parse_cfg_file(cfg_path):
    try:
        with open(cfg_path, "r", encoding="utf-8", errors="ignore") as f:
            return [line.strip() for line in f if line.strip()]
    except Exception:
        return []

# Fonction pour nettoyer et extraire l'assembly de la chaîne donnée
def clean_assembly_instruction(label):
    parts = label.split(":", 2)
    instruction = parts[2].strip() if len(parts) >= 3 else parts[-1].strip()
    assembly_instructions = ""  # Initialisation de la variable
    if instruction:
        mnemonic = MNEMONIC_CLEANER.sub('', instruction.split()[0]) if instruction else ""
        # Extraction des registres
        registers = REGISTER_REGEX.findall(instruction)
        # Combiner le mnémonique et les registres
        if mnemonic and registers:
            assembly_instructions = f"{mnemonic} {' '.join(registers)}"
        elif mnemonic:
            assembly_instructions = mnemonic
    return assembly_instructions

# Fonction pour extraire les embeddings FastText
def get_fasttext_embedding(label):
    if label in embedding_cache:
        return embedding_cache[label]
    # Utilisation du modèle personnalisé pour générer les embeddings
    embedding = model.get_sentence_vector(label)
    embedding_cache[label] = embedding
    return embedding

# Fonction pour extraire les features du CFG
def extract_features(name, directory):
    cfg_path = os.path.join(directory, f"{name}.json")

    if os.path.exists(cfg_path):
        try:
            G = nx.DiGraph()
            file = parse_cfg_file(cfg_path)

            node_labels = {}

            for line in file:
                if '->' in line:  # Extraction des arêtes
                    edge_match = EDGE_REGEX.match(line)
                    if edge_match:
                        src, dst = edge_match.groups()
                        G.add_edge(src, dst)

                if "[" in line and "]" in line:  # Extraction des nœuds
                    node_match = NODE_REGEX.match(line)
                    if node_match:
                        node, label = node_match.groups()
                        # Extraire et nettoyer l'instruction d'assembly
                        assembly_instruction = clean_assembly_instruction(label)
                        node_labels[node] = assembly_instruction
                    else:
                        assembly_instruction = ""

            # Extraire les embeddings avec tqdm
            for node, assembly in tqdm(node_labels.items(), desc="Processing FastText embeddings"):
                G.add_node(node, assembly_vector=get_fasttext_embedding(assembly))

        except Exception as e:
            print(f"Error processing {cfg_path}: {e}")
            return None

        return G
    else:
        print(f"File {cfg_path} does not exist.")
        return None

# Test
directory = "/Users/nouroudine/Documents/Challenge_Sorbonne_2025/folder_test_set"


In [ ]:
G = extract_features("0a00d1d1e996f7406ba1788851e68ce5a59a20a78370fa8c56162419365707b7", directory)

if G:
    print(f"Nombre de nœuds : {G.number_of_nodes()}")
    print(f"Nombre d'arêtes : {G.number_of_edges()}")
    first_node = list(G.nodes())[100]
    print(f"Embedding du premier nœud : {G.nodes[first_node]['assembly_vector'][:100]}")  # Affiche les 10 premiers éléments

In [9]:
import torch
from torch_geometric.data import Data
import networkx as nx

def convert_networkx_to_pyg(graph: nx.DiGraph):
    # Créer un mappage des nœuds vers des indices numériques
    node_mapping = {node: i for i, node in enumerate(graph.nodes())}
    
    # Extraire les informations des nœuds et des arêtes
    node_features = []
    for node in graph.nodes(data=True):
        # Si tu as des attributs, tu peux extraire des vecteurs spécifiques (par exemple, assembly_vector)
        node_features.append(node[1].get('assembly_vector', [0] * 100))  # Par exemple, vecteur de taille 10
    
    # Convertir les nœuds en tensor
    x = torch.tensor(node_features, dtype=torch.float)

    # Extraire les arêtes et mapper les nœuds à leurs indices numériques
    edge_index = []
    for src, dst in graph.edges():
        # Convertir les nœuds source et destination en indices numériques
        src_idx = node_mapping[src]  # Convertir le nœud source en indice numérique
        dst_idx = node_mapping[dst]  # Convertir le nœud destination en indice numérique
        edge_index.append([src_idx, dst_idx])

    # Convertir les arêtes en tensor de type long
    edge_index = torch.tensor(edge_index, dtype=torch.long)

    # Vérifier que l'edge_index a bien deux dimensions
    if edge_index.ndimension() == 1:
        edge_index = edge_index.unsqueeze(0)  # Si c'est un vecteur 1D, faire en sorte que ce soit 2D

    # Transposition et rendre le tensor contigu
    edge_index = edge_index.t().contiguous()

    # Optionnel : Ajouter des attributs pour les arêtes si nécessaire
    # edge_attr = torch.ones(edge_index.shape[1], 1)  # Valeur d'exemple, tu peux la personnaliser

    # Calcul des features globales
    num_nodes = len(graph.nodes())
    num_edges = len(graph.edges())
    avg_degree = sum(dict(graph.degree()).values()) / num_nodes if num_nodes > 0 else 0
    graph_density = num_edges / (num_nodes * (num_nodes - 1)) if num_nodes > 1 else 0
    connected_components = nx.number_weakly_connected_components(graph)
    centrality = nx.degree_centrality(graph)  # Calcul de la centralité des nœuds
    avg_centrality = sum(centrality.values()) / len(centrality) if len(centrality) > 0 else 0

    # Ajouter les features globales comme un vecteur
    global_features = torch.tensor([avg_degree, graph_density, connected_components, avg_centrality], dtype=torch.float)

    # Créer un objet PyG Data avec les features globales ajoutées
    data = Data(x=x, edge_index=edge_index, edge_attr=None, global_features=global_features)

    return data

In [ ]:
import pandas as pd
df = pd.read_csv("../test_set_metadata_to_predict.csv", sep=";")
graph_names = df['name'].values
labels = torch.tensor(df.drop(columns=['name']).values, dtype=torch.float)  # Shape [num_graphs, 453]

In [ ]:
graphs1 = []
for name in graph_names[:500]:
    graphs1.append(extract_features(name.strip(".json"), directory))

In [ ]:
dataset1 = []
for name, graph in zip(df['name'][:500], graphs1):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        dataset1.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset1, "graphs_pyg_test1.pkl")

In [ ]:
graphs2 = []
for name in graph_names[500:1000]:
    graphs2.append(extract_features(name.strip(".json"), directory))

In [ ]:
dataset2 = []
for name, graph in zip(df['name'][500:1000], graphs2):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        dataset2.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset2, "graphs_pyg_test2.pkl")

In [ ]:
graphs3 = []
for name in graph_names[1000:1500]:
    graphs3.append(extract_features(name.strip(".json"), directory))

In [ ]:
dataset3 = []
for name, graph in zip(df['name'][1000:1500], graphs3):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        dataset3.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset3, "graphs_pyg_test3.pkl")

In [ ]:
graphs4 = []
for name in graph_names[1500:2000]:
    graphs4.append(extract_features(name.strip(".json"), directory))

In [ ]:
dataset4 = []
for name, graph in zip(df['name'][1500:2000], graphs4):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        dataset4.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset4, "graphs_pyg_test_4.pkl")

In [ ]:
graphs5 = []
for name in graph_names[2000:2500]:
    graphs5.append(extract_features(name.strip(".json"), directory))


In [ ]:
dataset5 = []
for name, graph in zip(df['name'][2000:2500], graphs5):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        dataset5.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset5, "graphs_pyg_test_5.pkl")

In [ ]:
graphs6 = []
for name in graph_names[2500:3001]:
    graphs6.append(extract_features(name.strip(".json"), directory))

In [ ]:
from tqdm import tqdm

dataset6 = []
for name, graph in tqdm(zip(df['name'][2500:3001], graphs6), total=len(graphs6)):
    if graph is not None:
        data = convert_networkx_to_pyg(graph)
        dataset6.append(data)

from torch_geometric.data import Data
# Sauvegarder les objets Data
torch.save(dataset6, "graphs_pyg_test_6.pkl")

In [10]:
df = pd.read_csv("1000_assembly_feature.csv")
graph_names = df['name'].values
df

<IPython.core.display.Javascript object>

,name,64-bit execution via heavens gate,64bits,PEB access,accept command line arguments,access the Windows event log,act as TCP client,allocate RW memory,allocate RWX memory,allocate memory,allocate or change RW memory,allocate or change RWX memory,allocate thread local storage,android,anorganix,apatch,apk,arm,armadillo,aspack,asprotect,assembly,attach user process memory,attachment,authenticate HMAC,block operations on executable memory pages using Arbitrary Code Guard,bobsoft,bypass Windows File Protection,calculate modulo 256 via x86 assembly,calls-wmi,capture screenshot,capture screenshot in Go,cexe,change file permission on Linux,change memory protection,change the wallpaper,check HTTP status code,check OS version,check ProcessDebugFlags,check ProcessDebugPort,...,spreader,ssh-communication,start HTTP server,start TCP server,start minifilter driver,stealth,sudo,suspend thread,suspicious-dns,suspicious-udp,swap mouse buttons,switch active desktop,telephony,telnet-communication,telock,terminate process,terminate process via kill,terminate thread,themida,thinstall,trojan,trusted,tunneling,upack,upx,usb-autorun,validate payment card number using luhn algorithm,validate payment card number using luhn algorithm with lookup table,validate payment card number using luhn algorithm with no lookup table,via-tor,winzip,wise,worm,write and execute a file,write clipboard data,write file on Linux,write file on Windows,write pipe,xorcrypt,yoda
0,1b35c9dbf3cd9ac60015aaa6cd451c898defa6dac1ff43...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,0a4031ab00664cc5e202c8731798800f0475ef76800122...,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1a50a5aa16b1994d1d3212b192868f6e1299c71df657a6...,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
3,c8e592c3dbec667f85d6e311357590cd67407a403eb7c1...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,b69c8f405b8435b11becdb38782bdcd03b021b132182fd...,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
989,0d11b2f604660c5099ff72880f2be8215dbb9009a7e3df...,0,0,1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0
990,0cd24919806c6a034115ea35a4440611ac9a0277a24c16...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
991,0a4b7259c5a459697122a2c4e85d5499919fa6a9c91d66...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
992,0a7f54525b7baab792fbf9160bc27f3b6424b955dae0b5...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
graphs = []
for name in graph_names[:]:
    graphs1.append(extract_features(name.strip(".json"), directory))
    